# Lensing forecasts with a custom source catalog

This notebook shows how to use an external source catalog in SLSim. SLSim needs a redshift `z`, AB magnitudes `mag_<band>` (any band name) and, for Sersic sources, a size and an ellipticity. `prepare_source_catalog()` converts a catalog into this format.

As an example we use a sub-mm catalog with fluxes in mJy.

In [1]:
import numpy as np
from astropy import units as u
from astropy.table import Table
from astropy.cosmology import FlatLambdaCDM

import slsim.Pipelines as pipelines
from slsim.Sources.SourcePopulation.catalog_preparation import prepare_source_catalog
from slsim.Sources.SourcePopulation.galaxies import Galaxies
from slsim.Deflectors.DeflectorPopulation.galaxy_deflectors import GalaxyDeflectors
from slsim.Lenses.LensPopulation.lens_pop import LensPop
from slsim.Util.param_util import flux_to_ab_magnitude

np.random.seed(42)
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

        Use pytest instead. [astropy.tests.runner]
        Use pytest instead. [astropy.utils.decorators]


## External catalog

A toy catalog covering `catalog_area`. Replace it with your own, e.g. `Table.read("my_catalog.fits")`.

In [2]:
catalog_area = 1 * u.deg**2
n = 30000
s_150 = 0.1 * np.random.uniform(size=n) ** (-1 / 1.5)  # mJy
raw_catalog = Table(
    {
        "Z": np.random.uniform(1, 5, n),
        "S_90GHz": s_150 * (90 / 150) ** 3.5,
        "S_150GHz": s_150,
        "S_220GHz": s_150 * (220 / 150) ** 3.5,
    }
)

## Convert to SLSim conventions

Rename columns, convert fluxes into `mag_<band>`, and draw the missing sizes and ellipticities from the built-in models (see the `prepare_source_catalog` docstring). If your catalog has sizes or shapes, add them as columns instead.

In [3]:
catalog = prepare_source_catalog(
    raw_catalog,
    column_mapping={"Z": "z"},
    flux_columns={"S_90GHz": "90GHz", "S_150GHz": "150GHz", "S_220GHz": "220GHz"},
    flux_unit=u.mJy,
    fill_missing=["physical_size", "ellipticity"],
)
catalog[:3]

z,S_90GHz,S_150GHz,S_220GHz,mag_90GHz,mag_150GHz,mag_220GHz,physical_size,ellipticity
float64,float64,float64,float64,float64,float64,float64,float64,float64
3.5525782734886957,0.03220058885149455,0.19245732154307194,0.7353505363848267,20.13040608802585,18.189229528882734,16.73382958842564,0.9957230634531561,0.4854014341434691
2.8371698141367725,0.01730464483604916,0.10342685379946387,0.39517848325146576,20.804658896842646,18.86348233769953,17.40808239724244,1.1915088949056745,0.8382521566994167
4.857994099455763,0.020599549301463394,0.12311992497522586,0.47042275214211554,20.61542132597448,18.67424476683136,17.21884482637427,0.7641392059191456,0.5603413865183421


## Source and deflector populations

Use `catalog_type="external"` to keep the sizes as given. The deflectors are the standard SkyPy population.

In [4]:
sources = Galaxies(
    galaxy_list=catalog,
    cosmo=cosmo,
    sky_area=catalog_area,
    catalog_type="external",
    kwargs_cut={"band": "150GHz", "band_max": flux_to_ab_magnitude(0.1), "z_max": 6},
)

# can keep sky_area ~ 10 deg^2 here for fast generation of the deflector catalog.
# even if it's not the same as the source catalog, it will be fine as SLSim handles the area scaling internally!
skypy_pipeline = pipelines.SkyPyPipeline(skypy_config=None, sky_area=catalog_area, filters=None, cosmo=cosmo) 

deflectors = GalaxyDeflectors(
    red_galaxy_list=skypy_pipeline.red_galaxies,
    kwargs_cut={"band": "g", "band_max": 28, "z_max": 2.5},
    kwargs_mass2light={},
    cosmo=cosmo,
    sky_area=catalog_area,
)

## Draw lenses

Flux limits on the lensed source are converted to magnitude limits. A lens is kept if it is brighter than the limit in at least one band.

In [5]:
flux_limits = {"90GHz": 0.5, "150GHz": 1.0, "220GHz": 2.5}  # mJy, illustrative
survey_area = 10 * u.deg**2

lens_pop = LensPop(deflectors, sources, cosmo=cosmo, sky_area=survey_area)
lenses = lens_pop.draw_population(
    kwargs_lens_cuts={"mag_arc_limit": {band: flux_to_ab_magnitude(f) for band, f in flux_limits.items()}}
)
print(f"{len(lenses)} lenses in {survey_area}")

84 lenses in 10.0 deg2
